### Hello everyone and welcome to this new project! We’ll begin by using the OpenAI Agents SDK to build an agent that works with existing OpenAI tools like the code interpreter. Then, we’ll extend its abilities by integrating Tavily web search for structured, real-time results, and wrap that search function as a FunctionTool so it can be used directly within the SDK.

### Next, we’ll add multi-turn memory, allowing the agent to combine past conversation history with new search data for better reasoning. Finally, we’ll run a complete workflow that ties everything together: real-time search, persistent memory, and intelligent decision-making to showcase the agent’s full potential

In [ ]:
!pip install -q openai-agents==0.2.2 python-dotenv requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.1/161.1 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 150.7/150.7 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 234.6/234.6 kB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 948.6/948.6 kB 30.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 5.7 MB/s eta 0:00:00


In [ ]:
# Install the Tavily Python library
!pip install -q tavily-python

In [ ]:
import os
import requests
from openai import OpenAI
from dotenv import load_dotenv
from IPython.display import display, Markdown

from google.colab import userdata

openai_api_key = userdata.get("OPENAI_API_KEY")
os.environ["OPENAI_API_KEY"] = openai_api_key

tavily_api_key = userdata.get("TAVILY_API_KEY")
os.environ["TAVILY_API_KEY"] = tavily_api_key

print(f"Retrieved API Key: {openai_api_key[:5]}...{openai_api_key[-5:]}" if openai_api_key else "OpenAI API Key not found or is None")
openai_client = OpenAI(api_key = openai_api_key)
print("OpenAI Client successfully configured.")

# The Tavily client initialization and associated print statements are now handled
# in a separate cell (cell 86136d1b) to keep concerns separated and avoid NameErrors.

Retrieved API Key: sk-pr...c_bMA
OpenAI Client successfully configured.


In [ ]:
from tavily import TavilyClient

# Configure the Tavily client separately
tavily_client = TavilyClient(api_key=tavily_api_key)
print("Tavily Client successfully configured.")

Tavily Client successfully configured.


In [ ]:
def print_markdown(text: str):
    display(Markdown(text))

### **Define Tavily Search Function and create a tool**

In [ ]:
import json # import the json module for handling json data
from typing_extensions import TypedDict # import TypedDict for type hinting
from agents import function_tool

In [ ]:
# Define a TypedDict for the expected parameters for the tavily seach functions
# A typedDict is like a blueprint for a dictionary in python
# It tells python exactly what keys the dictionary should have and what type of values go with each key

class TavilySearchParams(TypedDict):
    query: str  # The Search Query String
    max_results: int # The maximum number of results to return


In [ ]:
import json # import the json module for handling json data
from typing_extensions import TypedDict # import TypedDict for type hinting
from agents import function_tool

# Let's define a function that searches the web using the Tavily API and gives back a short summary of the top results.
# Decorate the function as a FunctionTool for OpenAI Agents SDK.
@function_tool
def tavily_search(params: TavilySearchParams) -> str:
  """
  calls the Tavily API and returns a string summary of top search results

  Args:
  params (TavilySearchParams): Dictionary with 'query' (str) and 'max_results' (int).

  rerurns:
  str: A formatted string summarizing the top search results, or an error message.
  """

  # The web address (endpoint) for sending the search request to Tavily (Tavily API endpoint)
  url = "https://api.tavily.com/search" # Corrected 'seach' to 'search'

  # Tell the API that we are sending json data
  headers = {"Content-Type": "application/json"}

  # What we are sending to the API:
  # Our secret API key (so that Tavily will know it's us)
  # The search text (query)
  # How many results we want (defaults to 2 if not given)
  payload = {
      "api_key": tavily_api_key, # Use the Tavily API key from environment
      "query": params["query"], # The search query from params
      "max_results": params.get("max_results", 2), # Use max_results from params, default to 2 if not provided.
  }

  # Send the search results to tavily (POST means we are sending data)
  response = requests.post(url, json = payload, headers = headers)

  # Check if the search worked (200 is ok)
  if response.status_code == 200:  # If the request was successful
      results = response.json().get ("results", []) # Extract the results'list from the response json

      # Build a summary string with each result's title and content, numbered
      summary = "\n".join([f"{i+1}.{r['title']}: {r['content']}" for i, r in enumerate(results)]) # Return summary or fallback message
      return summary
  else:
      return f"Tavily API error: {response.status_code}" # Return error message with status code if request failed

## **Build and Run an AI Agent with Tavily Search Tool**

In [ ]:
# Let's add memory to our agent
from agents import SQLiteSession
session = SQLiteSession("live_researcher_practice")

In [ ]:
from agents import Agent

live_researcher_agent = Agent(name = "Live Market Researcher",
                              instructions = """

CONTEXT:
You are a world-class market research assistant with access to real-time web search via the tavily_search tool.

INSTRUCTION:
-Analyze the user's question and determine if recent or real-time information is needed.
-If the question involves recent events, news, or product info, always call tavily_search.
Summarize search results clearly and concisely, do not copy-paste.
Always start your answer with: "According to a web search"

INPUT:
You will receive a conversation history and the latest user question. Use the full context to inform your response.

OUTPUT:
Provide a clear, well-structured answer that references the search results when appropriate.
If you use tavily_search, integrate the findings into your summary
""",
    model = "gpt-4.1-mini",
    tools = [tavily_search])
print("Agent created with Tavily tool.")

Agent created with Tavily tool.


In [ ]:
from agents import Runner

# First Question
question1 = "What are people saying about the new GPT-5-Model?"

print_markdown(f"**User:** {question1}")
run1 = await Runner.run(
    starting_agent = live_researcher_agent,
    input = question1,
    session = session,
)
print_markdown(f"### Agent's answer\n'{run1.final_output}'")

**User:** What are people saying about the new GPT-5-Model?

### Agent's answer
'According to a web search, the new GPT-5 model has just been launched by OpenAI and is now available to all ChatGPT users, including free-tier users. People are generally impressed, noting that GPT-5 is smarter, faster, and significantly more useful across various fields such as writing, coding, and healthcare. OpenAI claims GPT-5 has a lower hallucination rate, meaning it fabricates less, and has undergone extensive safety testing, including 5,000 hours of evaluations.

The model introduces a feature called "safe completions," which provides high-level responses within safety limits rather than outright refusing potentially risky questions. OpenAI is offering three versions of the model—gpt-5, gpt-5-mini, and gpt-5-nano—catering to different cost and performance needs.

Initial user reports, like from professionals testing GPT-5 for accounting workflows, highlight strengths in providing high-quality, thoughtful outputs, although some niche tasks such as generating working Excel files may still perform better with other specialized modes.

Overall, the sentiment is positive, with users appreciating the advancements and usability improvements compared to GPT-4.'

## **Practice Opportunity**
### Use the live_researcher_agent AI to find and summarize the key features of your favorite vehicle, it can be electric or conventional.
### Example: "What do reviewers say about the new Cybertruck?"
### -Then test its memory and search ability with a follow-up: Summarize the main features

In [ ]:
from agents import Runner

session = SQLiteSession("live_researcher_practice")
prod_memory = []

question1 = "What do reviewews say about the new Cybertruck?"

print_markdown(f"**User:** {question1}")
run1 = await Runner.run(live_researcher_agent, question1, session = session)
print_markdown(f"**Answer:** {run1.final_output}")

**User:** What do reviewews say about the new Cybertruck?

**Answer:** According to a web search, reviews of the new Tesla Cybertruck are quite mixed and reflect a range of opinions:

1. Design and Styling: The Cybertruck's futuristic, Blade Runner-esque, and sharp-edged stainless steel design is highly polarizing. Some describe it as "really weird" and "dystopian," with Wall Street analysts suggesting it may remain a niche product rather than mainstream. However, its distinctive look also makes it a status symbol that turns heads, attracting celebrities and high-profile owners.

2. Market Performance: Tesla has produced Cybertrucks at a rate of 1,300 per week with plans to increase production. It has become the highest-selling EV pickup in the US recently, but there are concerns about demand slowing, evidenced by Tesla offering discounts and halting orders for the cheapest version.

3. Strategic Impact: Some analysts view the Cybertruck as a strategic misstep for Tesla due to the extensive resources spent on development and feel it diverted attention from lower-priced models Tesla now needs. Others believe it could be a major product launch, although it might not significantly influence Tesla's stock or financials.

4. Consumer Reception: While loyal Tesla fans continue to support the brand, traditional pickup truck owners may be disappointed due to the Cybertruck's unconventional style and features. The armored glass demo fail during the initial reveal also colored early perceptions.

5. Pricing and Options: The Cybertruck aims for a starting price near $50,000, but newer versions are priced higher, with price cuts offered to stimulate sales. The base rear-wheel-drive model was around $61,000 but was discontinued, leaving a $79,990 all-wheel-drive version as the cheapest.

In summary, the Cybertruck stands out for its unique futuristic design and status appeal but faces skepticism regarding broader market acceptance and strategic fit within Tesla's lineup. Its sales performance and Tesla's pricing strategies suggest demand dynamics remain fluid.

In [ ]:
question2 = "Summarize the mayn features in one paragraph"
print_markdown(f"**User:** {question2}")
run2 = await Runner.run(live_researcher_agent, question2, session = session)
print_markdown(f"**Answer** {run2.final_output}")

**User:** Summarize the mayn features in one paragraph

**Answer** The Tesla Cybertruck features a futuristic, angular stainless steel body with a bold, Blade Runner-inspired design that sets it apart visually from traditional pickups. It incorporates a full steer-by-wire system and Tesla's touted "Armor Glass" for enhanced durability, though the glass faced early skepticism. The vehicle offers various configurations, including rear-wheel and all-wheel drive, with ranges from around 250 to 325 miles per charge. Production takes place at Tesla's Texas Gigafactory, aiming for a starting price near $50,000, though current models are priced higher. Its design emphasizes both rugged utility and status appeal, targeting electric vehicle buyers looking for a distinctive, high-performance electric pickup.

## **Leveraging on Existing OpenAI Built-in Tools**
### Instead of creating a tool, you can leverage on existing tools available in OpenAI SDK.
### -WebSearchTool: Let your agent do a real-time web search.
### -FileSearchTool: Search your vector stores for content retrieval
### -ComputerTool: Have the agent take actions on your system like clicking or typing.
### -CodeInterpreterTool, ImageGenerationTool, HostedMCPTool: Extra built-ins for code, visuals and MCP connections.

In [ ]:
import json # import json module for handling the json data
from typing_extensions import TypedDict, Any # import "TypedDict" for type hinting, import "Any" for general typing.

# from agents import FunctionTool, function_tool, RunContextWrapper, CodeInterpreterTool.
from agents import FunctionTool, function_tool, CodeInterpreterTool # import agent tool decorators and helpers

# Use the hosted CodeInterpreterTool to provide a safe, sandboxed Python execution environment for the agent.
# This tool allows the agent to run Python code in a secure container, which is isolated from the main system.
# The tool_config specifies that we want an automatic container (per documentation)

code_interpreter = CodeInterpreterTool(
    tool_config = {"type": "code_interpreter",
                   "container": {"type":"auto"}}
)

# Print a message to confirm that all tools (get_catalog, run readonl_sql, code_interpreter) are ready for use.
print("Tool ready: code_interpreter")

Tool ready: code_interpreter


In [ ]:
from agents import Agent

analyst_agent = Agent(name = "Analyst Agent",
                      instructions = """


CONTEXT:
You are a world-class market research assistant with access to both real-time web search (via the tavily_search tool)
and Python code execution (the code_interpreter tool)

INSTRUCTION:
-Carefully analyze the user's question to determine whether it requires:
    -Recent or real-time information (use tavily_search).
    -Data analysis, calculations, or code execution (use coe_interpreter)
    -Or combination of both tools.
-Use tavily_search for up-to-date facts, news, or product information.
-Use code_interpreter for tasks involving data analysis, calculations, or code-based reasoning.
-If the task benefits from both tools, use them together and integrate the results.
-Clearly summarize your findings and reasoning. Do not copy-paste search results; always paraphrase.
-When using search, beging your answer with "According to a web search".
-When using code, explain your process and results clearly.

INPUT:
You will receive a conversation history and the latest user question. Use the full context to decide which tool(s) to use
and to inform your response.

OUTPUT:
Provide a clear, well-structured answer. Reference search results and/or code outputs as appropriate, and integrate them into your memory
""",
    model = "gpt-4.1-mini",
    tools = [tavily_search, code_interpreter],

)

print("Agent created with Tavily and Code Interpreter tools")

Agent created with Tavily and Code Interpreter tools


In [ ]:
session = SQLiteSession("live_researcher_practice_with_python")

# New question: Ask the agent to get the current price of a Tesla Cyber Truck in Canada.
# Then simulate the price increase if tariffs rise from 5% to 20% in 1% increments.
# And calculate the new prices using Python, providing a summary across all scenarios

question1 = (
          "Find the current price of a Tesla Cyber Truck in Canada"
          "Then simulate how the price will change if tariffs rise from 5% to 20% in 1% increments"
          "For each tariff rate, calculate the new prices using Python and provide a summary of the results across all scenarios"
)

print_markdown(f"**User:** {question1}")
run1 = await Runner.run(
    starting_agent = analyst_agent,
    input = question1,
    session = session,
)

print_markdown(f"### Agent's answer\n{run1.final_output}")

**User:** -Find the current price of a Tesla Cyber Truck in Canada-Then simulate how the price will change if tariffs rise from 5% to 20% in 1% increments-For each tariff rate, calculate the new prices using Python and provide a summary of the results across all scenarios

### Agent's answer
According to a web search, the current price of the Tesla Cybertruck in Canada for 2024 is approximately CAD 114,990 for the dual-motor AWD model after a recent price cut. The more expensive Cyberbeast version is around CAD 165,990, but for the simulation, I'll use the dual-motor AWD model price of CAD 114,990 as a base.

Next, I'll simulate how the price changes if tariffs increase from 5% to 20% in 1% increments, calculating the new prices for each tariff rate using Python.

In [ ]:
question4 = (
    "Please provide only the final simulated prices for the Tesla Cybertruck in Canada at the following exact tariff rates: "
    "5% tariff, 10% tariff, 15% tariff, and 20% tariff. List these four prices clearly."
)

print_markdown(f"**User:** {question4}")
run4 = await Runner.run(
    starting_agent=analyst_agent,
    input=question4,
    session=session,
)

print_markdown(f"### Agent's answer\n{run4.final_output}")

**User:** Please provide only the final simulated prices for the Tesla Cybertruck in Canada at the following exact tariff rates: 5% tariff, 10% tariff, 15% tariff, and 20% tariff. List these four prices clearly.

### Agent's answer
Here are the final simulated prices for the Tesla Cybertruck in Canada at the specified exact tariff rates:

- At 5% tariff: CAD 120,740
- At 10% tariff: CAD 125,490
- At 15% tariff: CAD 130,240
- At 20% tariff: CAD 134,990

Let me know if you need further details or analysis!